# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_05 — Merton Jump-Diffusion Model

### Research question

How does adding discontinuous jumps to geometric Brownian motion change return distributions, option prices, and implied volatility smiles?

This notebook extends the Black-Scholes-Merton framework by adding compound Poisson jumps to the stock-price process.

It covers:

1. motivation for jump-diffusion models;
2. Merton jump-diffusion dynamics;
3. jump compensator and risk-neutral drift correction;
4. simulation of jump-diffusion paths;
5. return distribution diagnostics;
6. comparison against geometric Brownian motion;
7. Merton's European option pricing series;
8. Monte Carlo validation;
9. implied volatility smile extraction;
10. sensitivity to jump intensity, jump mean, and jump volatility;
11. simple synthetic calibration;
12. limitations and modern extensions.

The key idea is:

> Brownian diffusion creates continuous paths and lognormal returns. Merton jump diffusion adds rare discontinuous moves, creating heavier tails and volatility smiles while retaining a tractable option-pricing formula.

## 1. Why add jumps?

The Black-Scholes-Merton model assumes that the stock price follows geometric Brownian motion:

$$
dS_t = \mu S_t dt + \sigma S_t dW_t
$$
This produces continuous sample paths and normally distributed log returns over fixed horizons.

But financial markets often exhibit:

- sudden news shocks;
- earnings gaps;
- macro announcements;
- crashes;
- limit moves;
- liquidity shocks;
- fat-tailed return distributions.

To model discontinuous moves, Merton introduced a jump component.

A jump-diffusion model combines:

1. a continuous Brownian diffusion;
2. a discontinuous compound Poisson jump process.

This allows the price path to move smoothly most of the time, but occasionally jump.

## 2. Merton jump-diffusion dynamics

Under the physical measure $\mathbb P$, a common Merton jump-diffusion specification is:

$$
\begin{aligned}
\frac{dS_t}{S_{t^-}} &= \mu dt \\
&\quad + \sigma dW_t \\
&\quad + (J-1)dN_t
\end{aligned}
$$
where:

- $S_{t^-}$ is the price just before a possible jump;
- $W_t$ is Brownian motion;
- $N_t$ is a Poisson process with intensity $\lambda$;
- $J$ is a positive random jump multiplier.

Merton assumes lognormal jumps:

$$
\log J \sim \mathcal N(\mu_J,\sigma_J^2)
$$
The expected jump multiplier is:

$$
\mathbb E[J] = e^{\mu_J + \frac{1}{2}\sigma_J^2}
$$
Define:

$$
\begin{aligned}
\kappa = \mathbb E[J-1] &= e^{\mu_J + \frac{1}{2}\sigma_J^2} - 1
\end{aligned}
$$
Under the risk-neutral measure $\mathbb Q$, the drift must be adjusted so that the discounted stock price is a martingale:

$$
\begin{aligned}
\frac{dS_t}{S_{t^-}} &= (r-q-\lambda\kappa)dt \\
&\quad + \sigma dW_t^{\mathbb Q} \\
&\quad + (J-1)dN_t
\end{aligned}
$$
The term:

$$
-\lambda\kappa
$$
is the jump compensator. It subtracts the expected jump contribution from the drift.

## 3. Exact log-price simulation

For a finite horizon $T$, the Merton model can be simulated as:

$$
\begin{aligned}
S_T &= S_0 \exp \Bigg[ (r-q-\lambda\kappa-\frac{1}{2}\sigma^2)T \\
&\quad + \sigma W_T \\
&\quad + \sum_{i=1}^{N_T}Y_i \Bigg]
\end{aligned}
$$
where:

$$
Y_i = \log J_i \sim \mathcal N(\mu_J,\sigma_J^2)
$$
and:

$$
N_T \sim \text{Poisson}(\lambda T)
$$
This form is useful for Monte Carlo simulation because it avoids discretisation error for terminal prices.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, factorial, log, pi, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class MertonJumpDiffusionParams:
    """
    Merton jump-diffusion model parameters.

    Parameters
    ----------
    s0:
        Initial asset price.

    risk_free_rate:
        Continuously compounded risk-free rate.

    dividend_yield:
        Continuous dividend yield.

    sigma:
        Diffusive volatility.

    jump_intensity:
        Expected number of jumps per year, lambda.

    jump_mean:
        Mean of log jump size, mu_J.

    jump_vol:
        Volatility of log jump size, sigma_J.
    """
    s0: float = 100.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.00
    sigma: float = 0.18
    jump_intensity: float = 0.75
    jump_mean: float = -0.08
    jump_vol: float = 0.22

    @property
    def jump_compensator(self) -> float:
        """
        kappa = E[J - 1] for lognormal jump multiplier J.
        """
        return exp(self.jump_mean + 0.5 * self.jump_vol ** 2) - 1.0

    @property
    def risk_neutral_log_drift(self) -> float:
        """
        Drift term in log S excluding Brownian and jump sums.
        """
        return (
            self.risk_free_rate
            - self.dividend_yield
            - self.jump_intensity * self.jump_compensator
            - 0.5 * self.sigma ** 2
        )


params = MertonJumpDiffusionParams()

pd.Series(asdict(params))

In [ ]:
pd.Series({
    "jump_compensator_kappa": params.jump_compensator,
    "risk_neutral_log_drift": params.risk_neutral_log_drift
})

## 5. Black-Scholes-Merton helper functions

We reuse BSM pricing and implied-volatility routines.

These help compare pure diffusion with jump diffusion.

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal CDF.
    """
    x = np.asarray(x)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal PDF.
    """
    x = np.asarray(x)
    return np.exp(-0.5 * x ** 2) / np.sqrt(2.0 * np.pi)


def bsm_price(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> float:
    """
    Black-Scholes-Merton European option price.
    """
    if maturity <= 0:
        if option_type == "call":
            return max(spot - strike, 0.0)
        if option_type == "put":
            return max(strike - spot, 0.0)
        raise ValueError("option_type must be 'call' or 'put'.")

    d1 = (
        log(spot / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * maturity
    ) / (volatility * sqrt(maturity))

    d2 = d1 - volatility * sqrt(maturity)

    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        return float(discounted_spot * normal_cdf(d1) - discounted_strike * normal_cdf(d2))

    if option_type == "put":
        return float(discounted_strike * normal_cdf(-d2) - discounted_spot * normal_cdf(-d1))

    raise ValueError("option_type must be 'call' or 'put'.")

## 6. Exact terminal simulation

We first simulate terminal prices directly.

For each path:

1. draw the number of jumps:

$$
N_T \sim \text{Poisson}(\lambda T)
$$
2. draw the total jump log-return:

$$
\sum_{i=1}^{N_T}Y_i
$$
3. add diffusion and drift terms.

This gives an exact terminal simulation under the Merton model.

In [ ]:
def simulate_merton_terminal_prices(
    params: MertonJumpDiffusionParams,
    maturity: float,
    n_paths: int,
    seed: int = 42
) -> pd.DataFrame:
    """
    Simulate terminal prices under Merton jump diffusion exactly at maturity.
    """
    local_rng = np.random.default_rng(seed)

    n_jumps = local_rng.poisson(
        lam=params.jump_intensity * maturity,
        size=n_paths
    )

    diffusion = params.sigma * sqrt(maturity) * local_rng.standard_normal(n_paths)

    jump_sum = np.empty(n_paths)

    for i, count in enumerate(n_jumps):
        if count == 0:
            jump_sum[i] = 0.0
        else:
            jump_sum[i] = local_rng.normal(
                loc=params.jump_mean,
                scale=params.jump_vol,
                size=count
            ).sum()

    log_return = (
        params.risk_neutral_log_drift * maturity
        + diffusion
        + jump_sum
    )

    terminal_price = params.s0 * np.exp(log_return)

    return pd.DataFrame({
        "terminal_price": terminal_price,
        "log_return": log_return,
        "n_jumps": n_jumps,
        "jump_log_return": jump_sum,
        "diffusion_log_return": diffusion
    })


terminal_sim = simulate_merton_terminal_prices(
    params=params,
    maturity=1.0,
    n_paths=200_000,
    seed=42
)

terminal_sim.head()

## 7. Return distribution diagnostics

Jump diffusion should produce heavier tails than pure geometric Brownian motion.

We compare simulated jump-diffusion log returns with pure diffusion log returns using:

- mean;
- volatility;
- skewness;
- excess kurtosis;
- quantiles;
- jump counts.

In [ ]:
def distribution_diagnostics(series: pd.Series) -> pd.Series:
    """
    Basic distribution diagnostics.
    """
    x = series.dropna().to_numpy()
    mean = np.mean(x)
    std = np.std(x, ddof=1)

    centered = x - mean
    skew = np.mean(centered ** 3) / (std ** 3)
    excess_kurtosis = np.mean(centered ** 4) / (std ** 4) - 3.0

    return pd.Series({
        "mean": mean,
        "std": std,
        "skew": skew,
        "excess_kurtosis": excess_kurtosis,
        "p01": np.quantile(x, 0.01),
        "p05": np.quantile(x, 0.05),
        "p50": np.quantile(x, 0.50),
        "p95": np.quantile(x, 0.95),
        "p99": np.quantile(x, 0.99)
    })


def simulate_gbm_terminal_log_returns(
    s0: float,
    risk_free_rate: float,
    dividend_yield: float,
    sigma: float,
    maturity: float,
    n_paths: int,
    seed: int = 123
) -> np.ndarray:
    """
    Simulate terminal log returns under pure GBM.
    """
    local_rng = np.random.default_rng(seed)

    return (
        (risk_free_rate - dividend_yield - 0.5 * sigma ** 2) * maturity
        + sigma * sqrt(maturity) * local_rng.standard_normal(n_paths)
    )


gbm_log_returns = simulate_gbm_terminal_log_returns(
    s0=params.s0,
    risk_free_rate=params.risk_free_rate,
    dividend_yield=params.dividend_yield,
    sigma=params.sigma,
    maturity=1.0,
    n_paths=len(terminal_sim),
    seed=123
)

diagnostic_table = pd.DataFrame({
    "merton_jump_diffusion": distribution_diagnostics(terminal_sim["log_return"]),
    "pure_gbm_same_sigma": distribution_diagnostics(pd.Series(gbm_log_returns))
})

diagnostic_table

In [ ]:
jump_count_distribution = terminal_sim["n_jumps"].value_counts(normalize=True).sort_index().reset_index()
jump_count_distribution.columns = ["n_jumps", "frequency"]

jump_count_distribution.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(terminal_sim["log_return"].clip(-1.0, 1.0), bins=120, density=True, alpha=0.6, label="Merton jump diffusion")
plt.hist(np.clip(gbm_log_returns, -1.0, 1.0), bins=120, density=True, alpha=0.5, label="Pure GBM")
plt.title("Terminal Log-Return Distribution")
plt.xlabel("Log return, clipped for display")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(jump_count_distribution["n_jumps"].astype(str), jump_count_distribution["frequency"])
plt.title("Simulated Number of Jumps over One Year")
plt.xlabel("Number of jumps")
plt.ylabel("Frequency")
plt.show()

## 8. Path simulation

Terminal simulation is enough for European option pricing, but path simulation is useful for visual intuition and later path-dependent options.

At each time step:

$$
\begin{aligned}
\Delta \log S &= (r-q-\lambda\kappa-\frac{1}{2}\sigma^2)\Delta t \\
&\quad + \sigma\sqrt{\Delta t}Z \\
&\quad + \sum_{i=1}^{\Delta N}Y_i
\end{aligned}
$$
where:

$$
\Delta N \sim \text{Poisson}(\lambda \Delta t)
$$

In [ ]:
def simulate_merton_paths(
    params: MertonJumpDiffusionParams,
    maturity: float,
    n_steps: int,
    n_paths: int,
    seed: int = 7
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Simulate full Merton jump-diffusion price paths.
    """
    local_rng = np.random.default_rng(seed)

    dt = maturity / n_steps
    time_grid = np.linspace(0.0, maturity, n_steps + 1)

    prices = np.empty((n_paths, n_steps + 1))
    jump_counts = np.zeros((n_paths, n_steps), dtype=int)

    prices[:, 0] = params.s0

    for step in range(1, n_steps + 1):
        dN = local_rng.poisson(params.jump_intensity * dt, size=n_paths)
        jump_counts[:, step - 1] = dN

        jump_sum = np.zeros(n_paths)

        jump_paths = np.where(dN > 0)[0]

        for idx in jump_paths:
            jump_sum[idx] = local_rng.normal(
                loc=params.jump_mean,
                scale=params.jump_vol,
                size=dN[idx]
            ).sum()

        diffusion = params.sigma * sqrt(dt) * local_rng.standard_normal(n_paths)

        log_increment = (
            params.risk_neutral_log_drift * dt
            + diffusion
            + jump_sum
        )

        prices[:, step] = prices[:, step - 1] * np.exp(log_increment)

    return time_grid, prices, jump_counts


time_grid, merton_paths, path_jump_counts = simulate_merton_paths(
    params=params,
    maturity=1.0,
    n_steps=252,
    n_paths=2_000,
    seed=2026
)

merton_paths.shape

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(60):
    plt.plot(time_grid, merton_paths[i], alpha=0.35)

plt.title("Merton Jump-Diffusion Sample Paths")
plt.xlabel("Time")
plt.ylabel("Price")
plt.show()

## 9. Merton European option pricing series

Merton's model has a semi-closed-form option pricing formula.

Conditional on $n$ jumps occurring by maturity, the log terminal price is normally distributed with:

- additional mean $n\mu_J$;
- additional variance $n\sigma_J^2$.

A convenient expression is a Poisson-weighted sum of Black-Scholes-style prices:

$$
\begin{aligned}
V^{MJD} &= \sum_{n=0}^{\infty} \Pr(N_T=n) V^{BSM}_n
\end{aligned}
$$
where:

$$
\begin{aligned}
\Pr(N_T=n) &= e^{-\lambda T} \frac{(\lambda T)^n}{n!}
\end{aligned}
$$
and $V^{BSM}_n$ is a Black-Scholes price with adjusted volatility and forward drift conditional on $n$ jumps.

This notebook implements the formula through the conditional lognormal distribution, which is transparent and numerically stable for moderate jump intensities.

In [ ]:
def bsm_price_from_lognormal_moments(
    spot: float,
    strike: float,
    maturity: float,
    discount_rate: float,
    mean_log_ST: float,
    variance_log_ST: float,
    option_type: str
) -> float:
    """
    Price a European option when log(S_T) is normally distributed.

    This is a general Black-Scholes-style expectation under a supplied
    lognormal terminal distribution and a discount rate.
    """
    std = sqrt(variance_log_ST)

    if std <= 0:
        terminal = exp(mean_log_ST)
        payoff = max(terminal - strike, 0.0) if option_type == "call" else max(strike - terminal, 0.0)
        return exp(-discount_rate * maturity) * payoff

    d2 = (mean_log_ST - log(strike)) / std
    d1 = d2 + std

    expected_ST_indicator = exp(mean_log_ST + 0.5 * variance_log_ST) * normal_cdf(d1)
    strike_indicator = strike * normal_cdf(d2)

    if option_type == "call":
        undiscounted = expected_ST_indicator - strike_indicator
    elif option_type == "put":
        undiscounted = (
            strike * normal_cdf(-d2)
            - exp(mean_log_ST + 0.5 * variance_log_ST) * normal_cdf(-d1)
        )
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    return float(exp(-discount_rate * maturity) * undiscounted)


def merton_option_price_series(
    params: MertonJumpDiffusionParams,
    strike: float,
    maturity: float,
    option_type: str,
    max_terms: int = 80,
    tolerance: float = 1e-14
) -> dict:
    """
    Price a European option under Merton jump diffusion using a Poisson mixture.
    """
    lambda_T = params.jump_intensity * maturity

    price = 0.0
    rows = []

    poisson_prob = exp(-lambda_T)

    for n in range(max_terms):
        if n == 0:
            poisson_prob = exp(-lambda_T)
        elif n > 0:
            poisson_prob *= lambda_T / n

        mean_log_ST = (
            log(params.s0)
            + params.risk_neutral_log_drift * maturity
            + n * params.jump_mean
        )

        variance_log_ST = (
            params.sigma ** 2 * maturity
            + n * params.jump_vol ** 2
        )

        conditional_price = bsm_price_from_lognormal_moments(
            spot=params.s0,
            strike=strike,
            maturity=maturity,
            discount_rate=params.risk_free_rate,
            mean_log_ST=mean_log_ST,
            variance_log_ST=variance_log_ST,
            option_type=option_type
        )

        contribution = poisson_prob * conditional_price
        price += contribution

        rows.append({
            "n_jumps": n,
            "poisson_probability": poisson_prob,
            "conditional_price": conditional_price,
            "contribution": contribution,
            "cumulative_price": price
        })

        if n > lambda_T + 10 and contribution < tolerance:
            break

    return {
        "price": float(price),
        "terms_used": len(rows),
        "term_table": pd.DataFrame(rows)
    }

In [ ]:
strike = 100.0
maturity = 1.0

merton_call = merton_option_price_series(
    params=params,
    strike=strike,
    maturity=maturity,
    option_type="call"
)

merton_put = merton_option_price_series(
    params=params,
    strike=strike,
    maturity=maturity,
    option_type="put"
)

bsm_call_same_diffusion = bsm_price(
    spot=params.s0,
    strike=strike,
    maturity=maturity,
    risk_free_rate=params.risk_free_rate,
    dividend_yield=params.dividend_yield,
    volatility=params.sigma,
    option_type="call"
)

pd.Series({
    "merton_call_price": merton_call["price"],
    "merton_put_price": merton_put["price"],
    "bsm_call_same_diffusive_sigma": bsm_call_same_diffusion,
    "merton_call_minus_bsm_call": merton_call["price"] - bsm_call_same_diffusion,
    "terms_used": merton_call["terms_used"]
})

In [ ]:
merton_call["term_table"].head(12)

In [ ]:
plt.figure(figsize=(10, 6))
term_table = merton_call["term_table"]
plt.bar(term_table["n_jumps"], term_table["contribution"])
plt.title("Poisson-Mixture Contributions to Merton Call Price")
plt.xlabel("Number of jumps")
plt.ylabel("Price contribution")
plt.show()

## 10. Monte Carlo validation

We validate the Merton pricing series using Monte Carlo simulation.

For each path:

$$
\text{payoff} = \max(S_T-K,0)
$$
The discounted Monte Carlo price is:

$$
e^{-rT}\frac{1}{M}\sum_{m=1}^{M}\text{payoff}^{(m)}
$$
The Monte Carlo estimate should be close to the analytical series price, with sampling error.

In [ ]:
def monte_carlo_merton_option_price(
    params: MertonJumpDiffusionParams,
    strike: float,
    maturity: float,
    option_type: str,
    n_paths: int = 300_000,
    seed: int = 123
) -> dict:
    """
    Monte Carlo price for a European option under Merton jump diffusion.
    """
    sim = simulate_merton_terminal_prices(
        params=params,
        maturity=maturity,
        n_paths=n_paths,
        seed=seed
    )

    ST = sim["terminal_price"].to_numpy()

    if option_type == "call":
        payoff = np.maximum(ST - strike, 0.0)
    elif option_type == "put":
        payoff = np.maximum(strike - ST, 0.0)
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    discounted_payoff = np.exp(-params.risk_free_rate * maturity) * payoff

    price = float(np.mean(discounted_payoff))
    standard_error = float(np.std(discounted_payoff, ddof=1) / np.sqrt(n_paths))

    return {
        "mc_price": price,
        "standard_error": standard_error,
        "lower_95": price - 1.96 * standard_error,
        "upper_95": price + 1.96 * standard_error,
        "n_paths": n_paths
    }


mc_call = monte_carlo_merton_option_price(
    params=params,
    strike=strike,
    maturity=maturity,
    option_type="call",
    n_paths=300_000,
    seed=2027
)

pd.Series({
    "merton_series_call": merton_call["price"],
    **mc_call,
    "series_minus_mc": merton_call["price"] - mc_call["mc_price"]
})

## 11. Put-call parity check

Merton jump diffusion under the risk-neutral measure still gives arbitrage-free European prices.

So put-call parity should hold:

$$
\begin{aligned}
C - P &= S_0e^{-qT} - Ke^{-rT}
\end{aligned}
$$
The jump component changes the distribution of $S_T$, but it does not break parity.

In [ ]:
put_call_parity_error = (
    merton_call["price"]
    - merton_put["price"]
    - (
        params.s0 * exp(-params.dividend_yield * maturity)
        - strike * exp(-params.risk_free_rate * maturity)
    )
)

pd.Series({
    "merton_call": merton_call["price"],
    "merton_put": merton_put["price"],
    "parity_error": put_call_parity_error
})

## 12. Implied volatility from Merton prices

A jump-diffusion model can generate a smile when its prices are interpreted through the BSM model.

For each strike, compute the Merton option price and invert the Black-Scholes formula to get BSM implied volatility.

If jump risk increases out-of-the-money put prices, the BSM implied volatility for low strikes will rise, creating skew/smile behaviour.

In [ ]:
def implied_vol_bisection(
    target_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    low: float = 1e-6,
    high: float = 5.0,
    tolerance: float = 1e-10,
    max_iter: int = 200
) -> float | None:
    """
    Simple robust implied volatility solver by bisection.
    """
    intrinsic_lower = (
        max(spot * exp(-dividend_yield * maturity) - strike * exp(-risk_free_rate * maturity), 0.0)
        if option_type == "call"
        else max(strike * exp(-risk_free_rate * maturity) - spot * exp(-dividend_yield * maturity), 0.0)
    )

    upper = spot * exp(-dividend_yield * maturity) if option_type == "call" else strike * exp(-risk_free_rate * maturity)

    if target_price < intrinsic_lower - 1e-10 or target_price > upper + 1e-10:
        return None

    def objective(vol):
        return bsm_price(
            spot=spot,
            strike=strike,
            maturity=maturity,
            risk_free_rate=risk_free_rate,
            dividend_yield=dividend_yield,
            volatility=vol,
            option_type=option_type
        ) - target_price

    f_low = objective(low)
    f_high = objective(high)

    if f_low * f_high > 0:
        return None

    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        f_mid = objective(mid)

        if abs(f_mid) < tolerance or (high - low) < tolerance:
            return float(mid)

        if f_low * f_mid <= 0:
            high = mid
            f_high = f_mid
        else:
            low = mid
            f_low = f_mid

    return float(0.5 * (low + high))

In [ ]:
def merton_implied_vol_smile(
    params: MertonJumpDiffusionParams,
    strikes: np.ndarray,
    maturity: float,
    option_type: str
) -> pd.DataFrame:
    """
    Compute BSM implied vols from Merton option prices across strikes.
    """
    rows = []

    for strike_value in strikes:
        price_result = merton_option_price_series(
            params=params,
            strike=float(strike_value),
            maturity=maturity,
            option_type=option_type
        )

        iv = implied_vol_bisection(
            target_price=price_result["price"],
            spot=params.s0,
            strike=float(strike_value),
            maturity=maturity,
            risk_free_rate=params.risk_free_rate,
            dividend_yield=params.dividend_yield,
            option_type=option_type
        )

        rows.append({
            "strike": float(strike_value),
            "maturity": maturity,
            "option_type": option_type,
            "merton_price": price_result["price"],
            "bsm_implied_vol": iv,
            "log_moneyness": log(strike_value / params.s0),
            "terms_used": price_result["terms_used"]
        })

    return pd.DataFrame(rows)


strike_grid = np.linspace(60, 140, 41)

call_smile_1y = merton_implied_vol_smile(
    params=params,
    strikes=strike_grid,
    maturity=1.0,
    option_type="call"
)

put_smile_1y = merton_implied_vol_smile(
    params=params,
    strikes=strike_grid,
    maturity=1.0,
    option_type="put"
)

call_smile_1y.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(call_smile_1y["strike"], call_smile_1y["bsm_implied_vol"], marker="o", label="Call IV from Merton prices")
plt.plot(put_smile_1y["strike"], put_smile_1y["bsm_implied_vol"], marker="x", label="Put IV from Merton prices")
plt.axhline(params.sigma, linestyle="--", label="Diffusive sigma")
plt.title("BSM Implied Volatility Smile Generated by Merton Jump Diffusion")
plt.xlabel("Strike")
plt.ylabel("BSM implied volatility")
plt.legend()
plt.show()

## 13. Term structure of jump-induced implied volatility

Jump effects are often strongest at short maturities.

A fixed jump size has a larger annualised impact over short horizons than over long horizons.

We compute implied volatility smiles across several maturities.

In [ ]:
maturity_grid = np.array([14, 30, 60, 90, 180, 365]) / 365.0

surface_rows = []

for T in maturity_grid:
    smile = merton_implied_vol_smile(
        params=params,
        strikes=strike_grid,
        maturity=float(T),
        option_type="call"
    )
    surface_rows.append(smile)

iv_surface = pd.concat(surface_rows, ignore_index=True)

iv_surface.head()

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in iv_surface.groupby("maturity"):
    plt.plot(group["strike"], group["bsm_implied_vol"], marker="o", label=f"T={T:.2f}y")

plt.axhline(params.sigma, linestyle="--", label="Diffusive sigma")
plt.title("Merton Jump-Diffusion Implied Vol Smiles by Maturity")
plt.xlabel("Strike")
plt.ylabel("BSM implied volatility")
plt.legend()
plt.show()

## 14. Sensitivity to jump intensity

The jump intensity $\lambda$ controls the expected number of jumps per year.

Increasing $\lambda$ generally increases tail risk and affects option prices, especially away from the money.

In [ ]:
def sensitivity_to_jump_intensity(
    base_params: MertonJumpDiffusionParams,
    lambdas: list[float],
    strike: float,
    maturity: float,
    option_type: str
) -> pd.DataFrame:
    """
    Price and IV sensitivity to jump intensity.
    """
    rows = []

    for lam in lambdas:
        p = MertonJumpDiffusionParams(
            s0=base_params.s0,
            risk_free_rate=base_params.risk_free_rate,
            dividend_yield=base_params.dividend_yield,
            sigma=base_params.sigma,
            jump_intensity=lam,
            jump_mean=base_params.jump_mean,
            jump_vol=base_params.jump_vol
        )

        price = merton_option_price_series(
            params=p,
            strike=strike,
            maturity=maturity,
            option_type=option_type
        )["price"]

        iv = implied_vol_bisection(
            target_price=price,
            spot=p.s0,
            strike=strike,
            maturity=maturity,
            risk_free_rate=p.risk_free_rate,
            dividend_yield=p.dividend_yield,
            option_type=option_type
        )

        rows.append({
            "jump_intensity": lam,
            "merton_price": price,
            "bsm_implied_vol": iv,
            "jump_compensator": p.jump_compensator
        })

    return pd.DataFrame(rows)


lambda_grid = [0.0, 0.25, 0.50, 0.75, 1.00, 1.50, 2.00]

lambda_sensitivity = sensitivity_to_jump_intensity(
    base_params=params,
    lambdas=lambda_grid,
    strike=80.0,
    maturity=0.5,
    option_type="put"
)

lambda_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(lambda_sensitivity["jump_intensity"], lambda_sensitivity["bsm_implied_vol"], marker="o")
plt.title("Implied Volatility Sensitivity to Jump Intensity")
plt.xlabel("Jump intensity λ")
plt.ylabel("BSM implied volatility")
plt.show()

## 15. Sensitivity to jump mean and jump volatility

The jump mean $\mu_J$ controls average jump direction.

The jump volatility $\sigma_J$ controls jump-size dispersion.

Negative jump means are especially important for equity index options because downside crash risk tends to steepen implied volatility skew.

In [ ]:
def jump_parameter_grid(
    base_params: MertonJumpDiffusionParams,
    jump_means: np.ndarray,
    jump_vols: np.ndarray,
    strike: float,
    maturity: float,
    option_type: str
) -> pd.DataFrame:
    """
    Compute prices and IVs over jump mean and jump volatility grid.
    """
    rows = []

    for mu_j in jump_means:
        for sig_j in jump_vols:
            p = MertonJumpDiffusionParams(
                s0=base_params.s0,
                risk_free_rate=base_params.risk_free_rate,
                dividend_yield=base_params.dividend_yield,
                sigma=base_params.sigma,
                jump_intensity=base_params.jump_intensity,
                jump_mean=float(mu_j),
                jump_vol=float(sig_j)
            )

            price = merton_option_price_series(
                params=p,
                strike=strike,
                maturity=maturity,
                option_type=option_type
            )["price"]

            iv = implied_vol_bisection(
                target_price=price,
                spot=p.s0,
                strike=strike,
                maturity=maturity,
                risk_free_rate=p.risk_free_rate,
                dividend_yield=p.dividend_yield,
                option_type=option_type
            )

            rows.append({
                "jump_mean": float(mu_j),
                "jump_vol": float(sig_j),
                "merton_price": price,
                "bsm_implied_vol": iv,
                "jump_compensator": p.jump_compensator
            })

    return pd.DataFrame(rows)


jump_mean_grid = np.linspace(-0.20, 0.05, 11)
jump_vol_grid = np.linspace(0.05, 0.40, 11)

jump_grid = jump_parameter_grid(
    base_params=params,
    jump_means=jump_mean_grid,
    jump_vols=jump_vol_grid,
    strike=80.0,
    maturity=0.5,
    option_type="put"
)

jump_grid.head()

In [ ]:
pivot_jump = jump_grid.pivot(index="jump_vol", columns="jump_mean", values="bsm_implied_vol")

plt.figure(figsize=(10, 6))
plt.imshow(
    pivot_jump.values,
    origin="lower",
    aspect="auto",
    extent=[
        jump_mean_grid.min(),
        jump_mean_grid.max(),
        jump_vol_grid.min(),
        jump_vol_grid.max()
    ]
)
plt.colorbar(label="BSM implied volatility")
plt.title("Put IV Sensitivity to Jump Mean and Jump Volatility")
plt.xlabel("Jump mean μ_J")
plt.ylabel("Jump volatility σ_J")
plt.show()

## 16. Synthetic calibration exercise

We now generate synthetic option prices from known Merton parameters and attempt a simple grid calibration.

A full production calibration would use a numerical optimiser.

Here we use a coarse grid over:

- jump intensity $\lambda$;
- jump mean $\mu_J$;
- jump volatility $\sigma_J$.

The objective is mean squared error between model prices and synthetic market prices.

In [ ]:
def generate_synthetic_merton_market_chain(
    true_params: MertonJumpDiffusionParams,
    strikes: np.ndarray,
    maturities: np.ndarray,
    option_type: str,
    seed: int = 99
) -> pd.DataFrame:
    """
    Generate synthetic market option prices from Merton model with small noise.
    """
    local_rng = np.random.default_rng(seed)

    rows = []

    for T in maturities:
        for K in strikes:
            price = merton_option_price_series(
                params=true_params,
                strike=float(K),
                maturity=float(T),
                option_type=option_type
            )["price"]

            noise = local_rng.normal(0.0, max(0.005, 0.002 * price))
            market_price = max(price + noise, 1e-6)

            rows.append({
                "strike": float(K),
                "maturity": float(T),
                "option_type": option_type,
                "market_price": market_price,
                "true_model_price": price
            })

    return pd.DataFrame(rows)


calibration_strikes = np.linspace(75, 125, 11)
calibration_maturities = np.array([30, 90, 180]) / 365.0

market_chain = generate_synthetic_merton_market_chain(
    true_params=params,
    strikes=calibration_strikes,
    maturities=calibration_maturities,
    option_type="call",
    seed=2028
)

market_chain.head()

In [ ]:
def merton_price_for_chain(
    candidate_params: MertonJumpDiffusionParams,
    chain: pd.DataFrame
) -> np.ndarray:
    """
    Compute Merton prices for an option chain.
    """
    prices = []

    for row in chain.itertuples():
        price = merton_option_price_series(
            params=candidate_params,
            strike=float(row.strike),
            maturity=float(row.maturity),
            option_type=str(row.option_type)
        )["price"]
        prices.append(price)

    return np.array(prices)


def coarse_grid_calibration(
    base_params: MertonJumpDiffusionParams,
    market_chain: pd.DataFrame,
    lambda_grid: list[float],
    jump_mean_grid: list[float],
    jump_vol_grid: list[float]
) -> pd.DataFrame:
    """
    Coarse grid search over jump parameters.
    """
    rows = []
    market_prices = market_chain["market_price"].to_numpy()

    for lam in lambda_grid:
        for mu_j in jump_mean_grid:
            for sig_j in jump_vol_grid:
                candidate = MertonJumpDiffusionParams(
                    s0=base_params.s0,
                    risk_free_rate=base_params.risk_free_rate,
                    dividend_yield=base_params.dividend_yield,
                    sigma=base_params.sigma,
                    jump_intensity=float(lam),
                    jump_mean=float(mu_j),
                    jump_vol=float(sig_j)
                )

                model_prices = merton_price_for_chain(candidate, market_chain)
                mse = float(np.mean((model_prices - market_prices) ** 2))
                mae = float(np.mean(np.abs(model_prices - market_prices)))

                rows.append({
                    "jump_intensity": float(lam),
                    "jump_mean": float(mu_j),
                    "jump_vol": float(sig_j),
                    "mse": mse,
                    "mae": mae
                })

    return pd.DataFrame(rows).sort_values("mse").reset_index(drop=True)


calibration_results = coarse_grid_calibration(
    base_params=params,
    market_chain=market_chain,
    lambda_grid=[0.25, 0.50, 0.75, 1.00, 1.25],
    jump_mean_grid=[-0.14, -0.10, -0.08, -0.06, -0.02],
    jump_vol_grid=[0.12, 0.18, 0.22, 0.28, 0.34]
)

calibration_results.head(10)

## 17. Calibration interpretation

The calibration exercise is intentionally simple.

Important points:

1. multiple parameter combinations can produce similar option prices;
2. jump intensity and jump size can trade off against each other;
3. calibration can be unstable when the option chain is sparse;
4. stochastic volatility and jumps can both create smiles;
5. calibration should include bid/ask errors, regularisation, and out-of-sample checks.

This is why Merton jump diffusion is useful pedagogically, but not always sufficient as a production smile model.

In [ ]:
best_calibration = calibration_results.iloc[0]

best_params = MertonJumpDiffusionParams(
    s0=params.s0,
    risk_free_rate=params.risk_free_rate,
    dividend_yield=params.dividend_yield,
    sigma=params.sigma,
    jump_intensity=float(best_calibration["jump_intensity"]),
    jump_mean=float(best_calibration["jump_mean"]),
    jump_vol=float(best_calibration["jump_vol"])
)

best_model_prices = merton_price_for_chain(best_params, market_chain)

calibration_fit = market_chain.copy()
calibration_fit["best_model_price"] = best_model_prices
calibration_fit["pricing_error"] = calibration_fit["best_model_price"] - calibration_fit["market_price"]

calibration_fit.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(calibration_fit["market_price"], calibration_fit["best_model_price"], alpha=0.8)
min_p = min(calibration_fit["market_price"].min(), calibration_fit["best_model_price"].min())
max_p = max(calibration_fit["market_price"].max(), calibration_fit["best_model_price"].max())
plt.plot([min_p, max_p], [min_p, max_p], linestyle="--")
plt.title("Synthetic Merton Calibration: Market vs Model Prices")
plt.xlabel("Synthetic market price")
plt.ylabel("Best grid model price")
plt.show()

## 18. Jump-risk diagnostics

The model parameters can be translated into intuitive quantities.

Expected number of jumps over horizon $T$:

$$
\mathbb E[N_T] = \lambda T
$$
Probability of at least one jump:

$$
\mathbb P(N_T \geq 1) = 1-e^{-\lambda T}
$$
Expected jump multiplier:

$$
\mathbb E[J] = e^{\mu_J+\frac{1}{2}\sigma_J^2}
$$
Expected jump return:

$$
\mathbb E[J-1] = \kappa
$$

In [ ]:
def jump_risk_summary(params: MertonJumpDiffusionParams, maturities: list[float]) -> pd.DataFrame:
    """
    Summarise jump risk over multiple horizons.
    """
    rows = []

    for T in maturities:
        rows.append({
            "maturity": T,
            "expected_number_of_jumps": params.jump_intensity * T,
            "probability_at_least_one_jump": 1 - exp(-params.jump_intensity * T),
            "expected_jump_multiplier": exp(params.jump_mean + 0.5 * params.jump_vol ** 2),
            "expected_jump_return_kappa": params.jump_compensator
        })

    return pd.DataFrame(rows)


jump_risk = jump_risk_summary(
    params=params,
    maturities=[1/252, 7/365, 30/365, 90/365, 180/365, 1.0, 2.0]
)

jump_risk

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(jump_risk["maturity"], jump_risk["probability_at_least_one_jump"], marker="o")
plt.title("Probability of at Least One Jump by Horizon")
plt.xlabel("Maturity in years")
plt.ylabel("Probability")
plt.show()

## 19. Saving model outputs

The notebook saves:

1. terminal return diagnostics;
2. Merton pricing series terms;
3. Monte Carlo validation;
4. implied volatility smiles and surface;
5. jump-parameter sensitivity tables;
6. synthetic calibration results;
7. jump-risk summary;
8. manifest.

In [ ]:
output_dir = Path("data/processed/merton_jump_diffusion_v1")
output_dir.mkdir(parents=True, exist_ok=True)

distribution_diagnostics_path = output_dir / "return_distribution_diagnostics.csv"
jump_count_path = output_dir / "jump_count_distribution.csv"
pricing_terms_path = output_dir / "merton_call_poisson_terms.csv"
mc_validation_path = output_dir / "monte_carlo_validation_call.csv"
call_smile_path = output_dir / "call_smile_1y.csv"
put_smile_path = output_dir / "put_smile_1y.csv"
iv_surface_path = output_dir / "merton_iv_surface.csv"
lambda_sensitivity_path = output_dir / "jump_intensity_sensitivity.csv"
jump_grid_path = output_dir / "jump_mean_vol_sensitivity.csv"
calibration_results_path = output_dir / "coarse_grid_calibration_results.csv"
calibration_fit_path = output_dir / "calibration_fit.csv"
jump_risk_path = output_dir / "jump_risk_summary.csv"
manifest_path = output_dir / "manifest.json"

diagnostic_table.to_csv(distribution_diagnostics_path)
jump_count_distribution.to_csv(jump_count_path, index=False)
merton_call["term_table"].to_csv(pricing_terms_path, index=False)
pd.DataFrame([mc_call]).to_csv(mc_validation_path, index=False)
call_smile_1y.to_csv(call_smile_path, index=False)
put_smile_1y.to_csv(put_smile_path, index=False)
iv_surface.to_csv(iv_surface_path, index=False)
lambda_sensitivity.to_csv(lambda_sensitivity_path, index=False)
jump_grid.to_csv(jump_grid_path, index=False)
calibration_results.to_csv(calibration_results_path, index=False)
calibration_fit.to_csv(calibration_fit_path, index=False)
jump_risk.to_csv(jump_risk_path, index=False)

manifest = {
    "dataset_name": "merton_jump_diffusion_outputs",
    "schema_version": "merton_jump_diffusion_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "params": asdict(params),
    "merton_call_price_strike_100_maturity_1": float(merton_call["price"]),
    "merton_put_price_strike_100_maturity_1": float(merton_put["price"]),
    "monte_carlo_call_price": float(mc_call["mc_price"]),
    "monte_carlo_call_standard_error": float(mc_call["standard_error"]),
    "calibration_best_params": asdict(best_params),
    "notes": [
        "Risk-neutral drift includes jump compensator -lambda*kappa.",
        "Terminal prices are simulated exactly using compound Poisson jumps.",
        "European option prices use a Poisson mixture of conditional lognormal prices.",
        "Implied volatilities are BSM implied vols backed out from Merton prices."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

manifest_path

## 20. Practical checklist for Merton jump diffusion

Before using a jump-diffusion model, check:

1. **Measure convention**  
   Are jump parameters physical $\mathbb P$ parameters or pricing $\mathbb Q$ parameters?

2. **Drift compensation**  
   Did you include $-\lambda\kappa$ in the risk-neutral drift?

3. **Jump distribution**  
   Are jumps lognormal, double-exponential, or something else?

4. **Calibration stability**  
   Are jump intensity and jump size separately identifiable from the option chain?

5. **Smile fit**  
   Does the model fit skew and term structure, or only one maturity?

6. **Risk management**  
   Does the model capture tail losses realistically?

7. **Hedging interpretation**  
   Jump risk cannot be perfectly hedged by continuous trading in the underlying.

8. **Simulation method**  
   Are terminal prices simulated exactly, or is there discretisation error?

9. **Numerical truncation**  
   Is the Poisson series truncated safely?

10. **Model comparison**  
   Does Merton outperform simpler BSM or more flexible stochastic-volatility models out of sample?

## 21. Limitations

### 21.1 Jump risk creates market incompleteness

In a pure BSM world, continuous trading in the stock and bond can replicate a European option.

With jumps, a sudden discontinuity cannot be hedged perfectly by continuous delta hedging.

This means jump risk may require additional risk premia or instruments.

### 21.2 Lognormal jumps are restrictive

Merton assumes normally distributed log-jump sizes.

Real crashes may have asymmetric, state-dependent, clustered, or regime-dependent jumps.

### 21.3 Constant jump intensity

The model assumes constant $\lambda$.

In reality, jump risk may rise around earnings, macro announcements, crises, or liquidity events.

### 21.4 Constant diffusive volatility

Merton jump diffusion adds jumps but keeps Brownian volatility constant.

Real option smiles often require both stochastic volatility and jumps.

### 21.5 Calibration can be unstable

Different combinations of jump intensity, jump mean, and jump volatility can generate similar option prices.

This creates parameter-identification problems.

### 21.6 European option focus

The semi-closed formula is for European options.

American options, barrier options, and path-dependent products require additional numerical methods.

### 21.7 Synthetic data

This notebook uses synthetic data and controlled pricing.

Real market calibration requires bid/ask filtering, interest-rate curves, dividend/forward inputs, and static-arbitrage checks.

## 22. Research frontier and current directions

Jump diffusion remains useful as a building block, but modern models often combine jumps with other features.

### 22.1 Bates model: stochastic volatility plus jumps

The Bates model combines Heston stochastic volatility with jumps.

This helps capture both:

- volatility clustering and skew from stochastic volatility;
- discontinuous crash risk from jumps.

A future notebook could compare BSM, Merton, Heston, and Bates calibrations on the same synthetic option surface.

### 22.2 Kou double-exponential jumps

Kou's jump-diffusion model uses asymmetric double-exponential jumps.

This can generate heavier downside tails and tractable option pricing.

A future notebook could compare lognormal Merton jumps with double-exponential Kou jumps.

### 22.3 Lévy models and infinite-activity jumps

Merton jumps are finite-activity: over finite time, the number of jumps is finite.

More advanced models use infinite-activity Lévy processes such as Variance Gamma, CGMY, or Normal Inverse Gaussian.

A future notebook could compare finite-activity and infinite-activity jump models.

### 22.4 Fourier transform pricing

Many jump and affine models have characteristic functions.

Fourier methods such as Carr-Madan FFT or Lewis inversion can price many strikes efficiently.

A future notebook could implement characteristic-function pricing for Merton jump diffusion and compare it with the Poisson series.

### 22.5 Jump detection from high-frequency data

With intraday or L1 data, one can try to separate continuous variation from jumps using realised variance and bipower variation.

A future notebook could connect this Merton model to the realised-volatility estimator notebooks.

### 22.6 Deep jump-diffusion and neural SDEs

Modern models may parameterise jump intensity or jump-size distributions using neural networks.

A future notebook could simulate state-dependent jump intensity and learn it from synthetic option prices or returns.

### 22.7 Risk management under jump tails

Jump models are important not only for option pricing but also for stress testing, VaR, expected shortfall, and crash hedging.

A future notebook could compare VaR under GBM, Student-t, Merton jump diffusion, and historical stress scenarios.

## 23. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_06_monte_carlo_gbm_and_fat_tails**  
   Comparing GBM, jump diffusion, and heavy-tailed return simulations.

2. **02_07_exotic_options_monte_carlo**  
   Pricing path-dependent options where jumps affect barrier crossing and path behaviour.

3. **02_08_dynamic_delta_hedging_simulation**  
   Showing why jump risk creates delta-hedging errors.

4. **02_11_heston_model_calibration**  
   Moving from jumps with constant volatility to stochastic volatility.

5. **02_Bates_stochastic_volatility_with_jumps**  
   Combining Heston volatility and Merton jumps.

6. **03_01_garch_volatility_modeling**  
   Comparing jump-induced fat tails with volatility clustering.

7. **04_06_var_cvar_and_stress_testing**  
   Using jump diffusion to model crash scenarios.

8. **07_02_volatility_surface_pricing_and_alpha**  
   Connecting jump risk to option surface skew and tail-risk signals.

## 24. Summary

This notebook implemented the Merton jump-diffusion model.

It showed how to:

1. add compound Poisson jumps to geometric Brownian motion;
2. compute the jump compensator:

$$
\kappa = e^{\mu_J+\frac{1}{2}\sigma_J^2}-1
$$
3. simulate terminal and pathwise jump-diffusion prices;
4. compare return distributions against GBM;
5. price European options using a Poisson mixture;
6. validate prices with Monte Carlo;
7. extract BSM implied volatility smiles from Merton prices;
8. study sensitivity to jump intensity, jump mean, and jump volatility;
9. run a simple synthetic calibration;
10. save outputs for later volatility-surface and risk notebooks.

The key mathematical takeaway is:

> Merton jump diffusion preserves much of the tractability of BSM while introducing discontinuous moves and heavier tails.

The key financial takeaway is:

> Jumps help explain crash risk and volatility smiles, but they also make markets incomplete and hedging less perfect.

## 25. Further reading

### Classical references

- Merton, R. C. "Option Pricing When Underlying Stock Returns Are Discontinuous."
- Black, F. and Scholes, M. "The Pricing of Options and Corporate Liabilities."
- Hull, J. C. *Options, Futures, and Other Derivatives.*

### Jump-diffusion extensions

- Bates, D. S. "Jumps and Stochastic Volatility: Exchange Rate Processes Implicit in Deutsche Mark Options."
- Kou, S. "A Jump-Diffusion Model for Option Pricing."
- Cont, R. and Tankov, P. *Financial Modelling with Jump Processes.*

### Numerical methods

- Poisson mixture pricing.
- Fourier transform option pricing.
- Monte Carlo simulation with jumps.
- PIDE methods for jump-diffusion models.
- Calibration with bid/ask and regularisation.

### Future extensions

- Bates stochastic volatility with jumps.
- Kou double-exponential jumps.
- Variance Gamma and CGMY Lévy models.
- Jump detection using realised variation.
- Deep jump-diffusion models.